In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/LLM-Email-Classifier

/content/drive/MyDrive/LLM-Email-Classifier


In [4]:
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

print("Setup complete!")

Setup complete!


In [5]:
# Load datasets (exact columns from Kaggle)

# Use the combined file from Naser's dataset
spamAssasin_df = pd.read_csv("data/SpamAssasin.csv", usecols=['subject', 'label'])
nigerianFraud_df = pd.read_csv("data/Nigerian_Fraud.csv", usecols=['subject', 'label'])



In [6]:
display(spamAssasin_df.sample(5),nigerianFraud_df.sample(5))

,subject,label
2592,[ILUG] putty + proxy goodness,0
2475,Re: New Sequences Window,0
1675,[Spambayes] Ditching WordInfo,0
3540,Re: Al'Qaeda's fantasy ideology: Policy Revie...,0
4194,ADV: Professional Teeth Whitening At Home! ...,1


,subject,label
805,CHARITY SUPPORT,1
28,Request,1
750,Mr.Larry Edwards.,1
2835,JUST ANNOUNCED!,1
3232,I AM WAITING TO RECEIVE YOUR MAIL‎,1


In [7]:
# Check the class imabalance to see which dataset is useful
print(spamAssasin_df['label'].value_counts())
print(nigerianFraud_df['label'].value_counts())

label
0    4091
1    1718
Name: count, dtype: int64
label
1    3332
Name: count, dtype: int64


In [8]:
# There is a class imbalance merging to of the datasets into one
train_df = pd.concat([spamAssasin_df, nigerianFraud_df], ignore_index=True)

In [9]:
train_df.shape

(9141, 2)

In [10]:
train_df.head()

,subject,label
0,Re: New Sequences Window,0
1,[zzzzteana] RE: Alexander,0
2,[zzzzteana] Moscow bomber,0
3,[IRR] Klez: The Virus That Won't Die,0
4,Re: [zzzzteana] Nothing like mama used to make,0


In [11]:
# check for null values in the subject and labels
print(train_df.isnull().sum())

subject    55
label       0
dtype: int64


In [12]:
# Checking whether these null subject lines falls to both classes
train_df[train_df['subject'].isnull()]['label'].value_counts()

,count
label,
1,51
0,4


In [13]:
# Since majority of the null subject lines falls to spam, will be removing the minority
# Reason since the model is going to rely on the subject line , the model might get confuse even though there is not a huge difference

train_df = train_df.loc[~(train_df['subject'].isnull() & (train_df['label'] == 0))]

In [14]:
train_df.shape

(9137, 2)

Attempting to finetune a LLM, for this subject and label must extracted and attach to a prompt. So that the LLM model will correctly adapt, thereby creating a seperate column known as text.

In [15]:
def create_classifier_example(row):
    subject = row['subject']
    verdict = "PHISHING" if row['label'] == 1 else "LEGITIMATE"
    prompt= f"""<|im_start|>system
    You are a precise phishing email subject classifier.
    Reply with ONLY one word: PHISHING or LEGITIMATE. Do not add any explanation.
    <|im_start|>user
    Analyze this email subject and classify it:

    Subject: {subject}
    <|im_start|>assistant
    {verdict}"""
    return prompt

In [16]:
train_df['text'] = train_df.apply(create_classifier_example, axis=1)

In [17]:
print(train_df['text'].iloc[5])
print("\n--- Next example ---\n")
print(train_df['text'].iloc[5607][:800] + "...")

<|im_start|>system
    You are a precise phishing email subject classifier.
    Reply with ONLY one word: PHISHING or LEGITIMATE. Do not add any explanation.
    <|im_start|>user
    Analyze this email subject and classify it:

    Subject: Re: [zzzzteana] Nothing like mama used to make
    <|im_start|>assistant
    LEGITIMATE

--- Next example ---

<|im_start|>system
    You are a precise phishing email subject classifier.
    Reply with ONLY one word: PHISHING or LEGITIMATE. Do not add any explanation.
    <|im_start|>user
    Analyze this email subject and classify it:

    Subject: =?big5?Q?=A7A=A6b=B4M=A7=E4=BE=F7=B7|=B6=DC=3F=3F=A5=B4=B6}=A8=D3=AC=DD=AC=DD?=
    <|im_start|>assistant
    PHISHING...


In [18]:
train_df.to_csv("data/train_df.csv", index=False)